# WHACS offshore waves at the goals

Reads WHACS (bulk and the four partitions: 0 = wind sea, 1-3 = swells) at the
WHACS node of every goal from the local copy of the global archive, cleans the
peak periods and writes the processed zarr used by the dynamic runs and the
reconstruction. The partitions are written as WHACS gives them: the energy they
miss is measured, and closed on the bulk, by the calibration (07b).

The archive is the `archive` of the WHACS `offshore_waves` entry: one folder per
variable with monthly files on the global SMC mesh, as CSIRO publishes them
([doi:10.25919/shdk-7p29](https://doi.org/10.25919/shdk-7p29)). Only the file
chunks that hold the goal nodes are read (`utils.whacs`).

**The archive is not part of HyGloW.** On the GeoOcean cluster it is
`/lustre/geocean/DATA/CSIRO/WHACS/global-smc`. Elsewhere, download the 20
variables read here (`hs`, `fp`, `dir`, `spr` and `phs`, `ptp`, `pdir`, `pspr`
0-3; about 14 TB for 1979-2026) or ask GeoOcean for the goal series of your case.

**Prerequisites:** `04_Select_Goals_by_Hindcasts.ipynb` (`<Case>_Goals.json`,
with a `whacs` node per goal).
**Next:** `07b_Satellite_Calibration.ipynb` validates (and calibrates) this
hindcast against satellite altimetry.

## Configuration


In [ ]:
import xarray as xr
from utils import whacs
from utils.config import get_offshore_wave, resolve

offshore_name = "WHACS"  # offshore_waves entry with the archive
period = None  # months to read, e.g. slice("1979", "2023"); None = the whole archive
workers = 16  # processes reading the monthly files

archive = whacs.archive_path(offshore_name)
print(f"archive: {archive}")
print(f"output : {resolve(get_offshore_wave(offshore_name)['data_path'])}")

archive: /lustre/geocean/DATA/CSIRO/WHACS/global-smc
output : /lustre/geocean/WORK/users/javiert/projects/HyGloW/Samoa/inputs/WHACS/processed/Samoa_WHACS_goal.zarr


## Goal nodes

The archive seapoint of each goal's WHACS node (goals JSON `hindcast.whacs`);
`distance` (degrees) is ~0 when the node is a WHACS seapoint.

In [2]:
whacs.goal_seapoints(archive, offshore_name)

,seapoint,longitude,latitude,distance
goal,,,,
1,970691,-173.21875,-13.46875,0.0
2,971044,-172.96875,-13.09375,0.0
3,971049,-172.34375,-13.09375,0.0
4,970815,-171.96875,-13.34375,0.0
5,970701,-171.46875,-13.46875,0.0
6,970364,-170.84375,-13.84375,0.0
7,970367,-170.46875,-13.84375,0.0
8,970142,-170.21875,-14.09375,0.0
9,969669,-170.21875,-14.59375,0.0


## Read the archive

Every variable of every month in `period` at the goal seapoints (`tp = 1 / fp`).

In [3]:
whacs_data = whacs.whacs_at_goals(offshore_name, period=period, workers=workers)
whacs_data

<xarray.Dataset> Size: 1GB
Dimensions:   (time: 412752, goal: 16)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2026-01-31T23:00:00
  * goal      (goal) int64 128B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
    seapoint  (goal) int64 128B 970691 971044 971049 ... 969892 970007 970352
Data variables: (12/21)
    hs        (time, goal) float64 53MB 2.16 2.236 2.26 ... 1.882 2.162 2.66
    fp        (time, goal) float64 53MB 0.064 0.064 0.064 ... 0.125 0.128 0.13
    dir       (time, goal) float64 53MB 338.3 342.0 344.2 ... 307.7 313.0 335.3
    spr       (time, goal) float64 53MB 48.8 48.5 44.0 43.1 ... 48.7 39.6 39.5
    phs0      (time, goal) float64 53MB nan nan nan nan ... 0.676 1.972 2.446
    ptp0      (time, goal) float64 53MB nan nan nan nan ... 7.55 6.96 7.84 7.65
    ...        ...
    pspr2     (time, goal) float64 53MB 18.6 24.0 10.8 19.8 ... 7.1 17.0 9.4 9.5
    phs3      (time, goal) float64 53MB 1.026 1.09 1.17 ... 0.418 0.334 0.44
    ptp3      (time, goal) float64 53MB 11.24 8.45 11.25 ... 13.42 7.68 13.48
    pdir3     (time, goal) float64 53MB 19.3 308.3 15.7 ... 218.6 92.5 216.1
    pspr3     (time, goal) float64 53MB 19.8 16.9 19.7 10.8 ... 10.1 16.3 8.3
    tp        (time, goal) float64 53MB 15.62 15.62 15.62 ... 8.0 7.812 7.692
Attributes: (12/54)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lat_max:              74.90625
    geospatial_lon_min:              -180.0
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_resolution:        P0DT1H0M0S
    hywaves_source:                  WHACS global archive /lustre/geocean/DAT...

Bulk `tp = 1 / fp` explodes when the spectral peak falls below the WHACS
frequency grid (hours with Tp of 50-250 s). Those take the peak period of the
most energetic partition (`clean_peak_periods`); partition periods above
`max_tp` become NaN.

In [ ]:
from bluemath_tk.waves.hywaves.offshore import clean_peak_periods

max_tp = 25.0  # s, largest valid peak period
whacs_data = clean_peak_periods(whacs_data, max_tp=max_tp)
print(
    f"bulk tp replaced: {whacs_data.attrs['tp_replaced']}, partition tp removed: {whacs_data.attrs['ptp_removed']}"
)

bulk tp replaced: 28, partition tp removed: 0


## Write the zarr

In [5]:
whacs_data.to_zarr(resolve(get_offshore_wave(offshore_name)["data_path"]), mode="w")

In [6]:
xr.open_zarr(resolve(get_offshore_wave(offshore_name)["data_path"])).load()

<xarray.Dataset> Size: 1GB
Dimensions:   (time: 412752, goal: 16)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2026-01-31T23:00:00
  * goal      (goal) int64 128B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
    seapoint  (goal) int64 128B 970691 971044 971049 ... 969892 970007 970352
Data variables: (12/21)
    dir       (time, goal) float64 53MB 338.3 342.0 344.2 ... 307.7 313.0 335.3
    fp        (time, goal) float64 53MB 0.064 0.064 0.064 ... 0.125 0.128 0.13
    hs        (time, goal) float64 53MB 2.16 2.236 2.26 ... 1.882 2.162 2.66
    pdir0     (time, goal) float64 53MB nan nan nan nan ... 20.7 313.7 340.3
    pdir1     (time, goal) float64 53MB 341.7 341.2 309.1 ... 299.6 332.1 339.2
    pdir2     (time, goal) float64 53MB 312.4 15.5 340.7 ... 78.7 218.1 287.6
    ...        ...
    ptp0      (time, goal) float64 53MB nan nan nan nan ... 7.55 6.96 7.84 7.65
    ptp1      (time, goal) float64 53MB 16.11 15.87 8.38 ... 8.11 13.9 13.82
    ptp2      (time, goal) float64 53MB 8.45 11.17 15.65 ... 7.69 13.44 9.49
    ptp3      (time, goal) float64 53MB 11.24 8.45 11.25 ... 13.42 7.68 13.48
    spr       (time, goal) float64 53MB 48.8 48.5 44.0 43.1 ... 48.7 39.6 39.5
    tp        (time, goal) float64 53MB 15.62 15.62 15.62 ... 8.0 7.812 7.692
Attributes: (12/56)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_resolution:        P0DT1H0M0S
    hywaves_source:                  WHACS global archive /lustre/geocean/DAT...
    tp_replaced:                     28
    ptp_removed:                     0